# 14 — Silver — Contratual (Compras.gov + PNCP)

## Purpose
Build `silver_compras` and `silver_pncp` — conformed contract tables from
federal and national procurement portals.
These tables feed the Gold contractual facts and the serving layer.

## Input
| Source | Path | Bronze rows |
|---|---|---|
| Compras.gov | `data/bronze/compras_contratos/` | ~761,370 |
| PNCP | `data/bronze/pncp_contratos/` | ~644,053 |

## Output
| Table | Path | Grain | Expected rows |
|---|---|---|---|
| `silver_compras` | `data/silver/silver_compras/` | 1 row per contractual event | ~712k PJ |
| `silver_pncp` | `data/silver/silver_pncp/data.parquet` | 1 row per contract | ~619k PJ+BRA |

## Steps
1. Imports and configuration
2. Key findings from EDA
3. Build silver_compras
4. Build silver_pncp
5. Validation

## Notes
- `silver_compras` filter: `length(niFornecedor) = 14 AND contratoExcluido = 'false'`
- `silver_pncp` filter: `tipoPessoa = 'PJ' AND (codigoPaisFornecedor = 'BRA' OR codigoPaisFornecedor IS NULL) AND length(niFornecedor) = 14`
- `cnpj_normalized` = `niFornecedor` directly — already clean 14-digit, no punctuation
- `cnpj_basico` = `LEFT(niFornecedor, 8)`
- Dates compras: ISO timestamp `YYYY-MM-DDTHH:MM:SS` → `LEFT(val, 10)::DATE`
- Dates pncp: `YYYY-MM-DD` → `CAST AS DATE` | `dataPublicacaoPncp` has timestamp → `LEFT(val, 10)`
- Values: US decimal (dot separator) → `CAST AS DECIMAL(15,2)` directly
- Negative `valorGlobal`: preserved + `_valor_negativo BOOLEAN` flag (contractual reductions)
- `unidadesRequisitantes`: kept as VARCHAR (inconsistent: some JSON, some free text)
- `silver_compras` partitioned by `_year_month` (historical queries by period)
- `silver_pncp` single file (644k rows — sphere partitions would be too small)
- ADR-002: CNPJ always VARCHAR — never cast to INTEGER

## Step 1 — Imports and configuration

In [ ]:
# =============================================================================
# Step 1 — Imports and configuration
# =============================================================================

import sys
import shutil
from pathlib import Path
from datetime import datetime, timezone

# --- utils -------------------------------------------------------------------
try:
    _nb_file = Path(__vsc_ipynb_file__).resolve()
    _root_candidate = _nb_file.parent.parent
except NameError:
    _root_candidate = Path.cwd().parent

sys.path.insert(0, str(_root_candidate / "utils"))

from paths        import get_project_root, bronze_path, silver_path, ensure_dir, to_sql_path
from duckdb_utils import get_connection, scalar
from validation   import CheckSuite

import duckdb

# --- Paths -------------------------------------------------------------------
ROOT = get_project_root()

# Bronze — glob para leitura (particionado por _year_month)
BD_COMPRAS = bronze_path(ROOT, "compras_contratos", glob=True)
BD_PNCP    = bronze_path(ROOT, "pncp_contratos",    glob=True)

# Silver — Path para escrita, string SQL para leitura
OUT_COMPRAS = ROOT / "data" / "silver" / "silver_compras"
OUT_PNCP    = ROOT / "data" / "silver" / "silver_pncp"

SD_COMPRAS = to_sql_path(OUT_COMPRAS)           # leitura com glob no Step 5
SD_PNCP    = silver_path(ROOT, "silver_pncp")   # arquivo único — data.parquet

STARTED_AT = datetime.now(timezone.utc)

# Idempotent — sempre reconstrói
for out_dir in [OUT_COMPRAS, OUT_PNCP]:
    if out_dir.exists():
        shutil.rmtree(out_dir)
    ensure_dir(out_dir)

# --- Connection --------------------------------------------------------------
con = get_connection()

print(f"ROOT         : {ROOT}")
print(f"BD_COMPRAS   : {BD_COMPRAS}")
print(f"BD_PNCP      : {BD_PNCP}")
print(f"OUT_COMPRAS  : {OUT_COMPRAS}")
print(f"OUT_PNCP     : {OUT_PNCP}")
print(f"Started at   : {STARTED_AT.isoformat()}")
print(f"DuckDB ver.  : {duckdb.__version__}")
print()
print("Configuration ready.")

In [ ]:
# =============================================================================
# Step 2 — Key findings from EDA
# =============================================================================

# Compras.gov vs PNCP
# ┌──────────────────────────┬──────────────────────────────┬─────────────────────────┐
# │ Aspect                   │ Compras.gov                  │ PNCP                    │
# ├──────────────────────────┼──────────────────────────────┼─────────────────────────┤
# │ Coverage                 │ Federal only                 │ All spheres (M/E/F/D)   │
# │ Period                   │ 2021-01 to 2026-04           │ 2026-01 to 2026-04      │
# │ Grain                    │ 1 row per contractual event  │ 1 row per contract       │
# │ Supplier field           │ niFornecedor (14-digit CNPJ) │ niFornecedor (14-digit)  │
# │ Date format              │ ISO timestamp                │ YYYY-MM-DD or timestamp  │
# │ Value format             │ US decimal (dot)             │ US decimal (dot)         │
# │ Negative values          │ 136 in valorGlobal           │ 0                        │
# └──────────────────────────┴──────────────────────────────┴─────────────────────────┘
#
# Grain difference — Compras.gov
#   Multiple rows per contract — each addendum, value change, or signing
#   is a separate row. numeroContrato + codigoOrgao is NOT unique.
#   Silver preserves all rows. Aggregation happens in Gold.
#
# Negative valorGlobal — Compras.gov
#   136 rows with negative valorGlobal are contractual reductions (addenda).
#   Preserved with _valor_negativo = TRUE flag for downstream filtering.
#
# PNCP esfera 'N'
#   7% of records have orgaoEntidade_esferaId = 'N' — poderId misuse
#   by some municipal systems. Preserved with _esfera_valida = FALSE flag.
#
# PNCP tipoPessoa = 'PJ' does not guarantee 14-digit CNPJ
#   406 records classified as PJ but carrying CPF (11 digits) — source error.
#   Filter: AND length(niFornecedor) = 14 removes them.

print("Step 2 — EDA findings loaded (no execution needed).")

## Step 3 — Build silver_compras

Transformation rules:
- Filter: `length(niFornecedor) = 14 AND contratoExcluido = 'false'`
- `cnpj_normalized` = `niFornecedor` directly (already clean)
- `cnpj_basico` = `LEFT(niFornecedor, 8)`
- Dates: `CAST(LEFT(val, 10) AS DATE)` — strip T00:00:00 timestamp
- `dataVigenciaFinal`: nullable (7,435 nulls — contracts with no end date)
- Values: `CAST(val AS DECIMAL(15,2))` — US decimal, no replace needed
- `_valor_negativo`: `valorGlobal < 0` flag
- `contratoExcluido`: `'true'/'false'` → BOOLEAN (kept for auditability even after filter)
- `_fonte`: literal `'compras_gov'`
- Partition by `_year_month`

In [ ]:
# =============================================================================
# Step 3 — Build silver_compras
# =============================================================================

t0 = datetime.now()

sql_compras = f"""
    COPY (
        SELECT
            -- CNPJ fields (ADR-002: always VARCHAR)
            niFornecedor                                                AS cnpj_normalized,
            LEFT(niFornecedor, 8)                                       AS cnpj_basico,
            nomeRazaoSocialFornecedor                                   AS nome_fornecedor,

            -- Organ
            codigoOrgao                                                 AS codigo_orgao,
            nomeOrgao                                                   AS nome_orgao,
            codigoUnidadeGestora                                        AS codigo_unidade_gestora,
            nomeUnidadeGestora                                          AS nome_unidade_gestora,

            -- Contract identifiers
            numeroContrato                                              AS numero_contrato,
            numeroCompra                                                AS numero_compra,
            numeroControlePncpContrato                                  AS numero_controle_pncp,
            idCompra                                                    AS id_compra,
            processo,

            -- Contract type
            codigoModalidadeCompra                                      AS codigo_modalidade,
            nomeModalidadeCompra                                        AS nome_modalidade,
            codigoTipo                                                  AS codigo_tipo,
            nomeTipo                                                    AS nome_tipo,
            codigoCategoria                                             AS codigo_categoria,
            nomeCategoria                                               AS nome_categoria,
            objeto,

            -- Dates: ISO timestamp -> DATE (strip T00:00:00)
            CAST(LEFT(dataVigenciaInicial, 10) AS DATE)                AS data_vigencia_inicial,
            CASE WHEN dataVigenciaFinal IS NULL OR dataVigenciaFinal = '' THEN NULL
                 ELSE CAST(LEFT(dataVigenciaFinal, 10) AS DATE)
            END                                                         AS data_vigencia_final,
            CAST(LEFT(dataHoraInclusao, 10) AS DATE)                   AS data_inclusao,

            -- Values: US decimal -> DECIMAL(15,2)
            TRY_CAST(valorGlobal     AS DECIMAL(15,2))                 AS valor_global,
            TRY_CAST(valorParcela    AS DECIMAL(15,2))                 AS valor_parcela,
            TRY_CAST(valorAcumulado  AS DECIMAL(15,2))                 AS valor_acumulado,

            -- Flags
            TRY_CAST(valorGlobal AS DECIMAL(15,2)) < 0                AS _valor_negativo,
            contratoExcluido = 'true'                                  AS contrato_excluido,

            -- Additional fields
            unidadesRequisitantes                                       AS unidades_requisitantes,

            -- Source tracking
            'compras_gov'                                               AS _fonte,
            _year_month,
            CURRENT_TIMESTAMP                                           AS _silver_loaded_at

        FROM read_parquet('{BD_COMPRAS}', hive_partitioning=true)
        WHERE length(niFornecedor) = 14
          AND contratoExcluido = 'false'
    )
    TO '{SD_COMPRAS}'
    (FORMAT PARQUET, PARTITION_BY (_year_month), OVERWRITE_OR_IGNORE TRUE)
"""

con.execute(sql_compras)

elapsed    = (datetime.now() - t0).total_seconds()
files      = list(OUT_COMPRAS.rglob("*.parquet"))
total_mb   = sum(f.stat().st_size for f in files) / (1024 * 1024)
n          = scalar(con, f"SELECT count(*) FROM read_parquet('{SD_COMPRAS}/**/*.parquet')")
partitions = sorted(set(f.parent.name for f in files))

print(f"silver_compras built in {elapsed:.1f}s")
print(f"Rows       : {n:,}")
print(f"Files      : {len(files)}")
print(f"Partitions : {len(partitions)}")
print(f"Size       : {total_mb:.1f} MB")

## Step 4 — Build silver_pncp

Transformation rules:
- Filter: `tipoPessoa = 'PJ' AND (codigoPaisFornecedor = 'BRA' OR codigoPaisFornecedor IS NULL)`
- `cnpj_normalized` = `niFornecedor` directly (already clean 14-digit)
- `cnpj_basico` = `LEFT(niFornecedor, 8)`
- Simple dates (YYYY-MM-DD): `CAST AS DATE` directly
- `dataPublicacaoPncp` has timestamp: `CAST(LEFT(val,10) AS DATE)`
- Values: `CAST AS DECIMAL(15,2)` — US decimal, no replace needed
- `orgaoEntidade_esferaId = 'N'` flagged as invalid (`_esfera_valida = FALSE`)
- `numeroRetificacao`: kept as VARCHAR — each retification already has unique numeroControlePNCP
- `_fonte`: literal `'pncp'`
- No partition: 644k rows, single file

In [ ]:
# =============================================================================
# Step 4 — Build silver_pncp
# =============================================================================

t0 = datetime.now()

sql_pncp = f"""
    COPY (
        SELECT
            -- CNPJ fields (ADR-002: always VARCHAR)
            niFornecedor                                                AS cnpj_normalized,
            LEFT(niFornecedor, 8)                                       AS cnpj_basico,
            nomeRazaoSocialFornecedor                                   AS nome_fornecedor,
            tipoPessoa                                                  AS tipo_pessoa,
            codigoPaisFornecedor                                        AS codigo_pais_fornecedor,

            -- Contract identifiers
            numeroControlePNCP                                          AS numero_controle_pncp,
            numeroControlePncpCompra                                    AS numero_controle_pncp_compra,
            numeroContratoEmpenho                                       AS numero_contrato_empenho,
            sequencialContrato                                          AS sequencial_contrato,
            anoContrato                                                 AS ano_contrato,
            numeroRetificacao                                           AS numero_retificacao,
            processo,

            -- Contract content
            objetoContrato                                              AS objeto,
            informacaoComplementar                                      AS informacao_complementar,

            -- Dates: YYYY-MM-DD -> DATE | dataPublicacaoPncp has timestamp
            CAST(dataAssinatura AS DATE)                               AS data_assinatura,
            CAST(dataVigenciaInicio AS DATE)                           AS data_vigencia_inicio,
            CAST(dataVigenciaFim AS DATE)                              AS data_vigencia_fim,
            CAST(LEFT(dataPublicacaoPncp, 10) AS DATE)                 AS data_publicacao_pncp,
            CAST(LEFT(dataAtualizacao, 10) AS DATE)                    AS data_atualizacao,

            -- Values: US decimal -> DECIMAL(15,2)
            TRY_CAST(valorInicial   AS DECIMAL(15,2))                  AS valor_inicial,
            TRY_CAST(valorGlobal    AS DECIMAL(15,2))                  AS valor_global,
            TRY_CAST(valorParcela   AS DECIMAL(15,2))                  AS valor_parcela,
            TRY_CAST(valorAcumulado AS DECIMAL(15,2))                  AS valor_acumulado,

            -- Contracting organ
            orgaoEntidade_cnpj                                          AS orgao_cnpj,
            orgaoEntidade_razaoSocial                                   AS orgao_razao_social,
            orgaoEntidade_poderId                                        AS orgao_poder,
            orgaoEntidade_esferaId                                       AS orgao_esfera,
            orgaoEntidade_esferaId IN ('F', 'E', 'M', 'D')             AS _esfera_valida,

            -- Contracting unit
            unidadeOrgao_codigoUnidade                                  AS unidade_codigo,
            unidadeOrgao_nomeUnidade                                    AS unidade_nome,
            unidadeOrgao_ufSigla                                        AS unidade_uf,
            unidadeOrgao_municipioNome                                  AS unidade_municipio,

            -- Contract type
            tipoContrato_id                                             AS tipo_contrato_id,
            tipoContrato_nome                                           AS tipo_contrato_nome,
            categoriaProcesso_id                                        AS categoria_processo_id,
            categoriaProcesso_nome                                      AS categoria_processo_nome,

            -- Source tracking
            'pncp'                                                      AS _fonte,
            CURRENT_TIMESTAMP                                           AS _silver_loaded_at

        FROM read_parquet('{BD_PNCP}')
        WHERE tipoPessoa = 'PJ'
            AND (codigoPaisFornecedor = 'BRA' OR codigoPaisFornecedor IS NULL)
            AND length(niFornecedor) = 14
            AND dataAssinatura BETWEEN '2021-01-01' AND '2030-12-31'
    )
    TO '{SD_PNCP}'
    (FORMAT PARQUET)
"""

con.execute(sql_pncp)
con.close()

elapsed = (datetime.now() - t0).total_seconds()
n       = scalar(get_connection(), f"SELECT count(*) FROM read_parquet('{SD_PNCP}')")
size_mb = (OUT_PNCP / "data.parquet").stat().st_size / (1024 * 1024)

print(f"silver_pncp built in {elapsed:.1f}s")
print(f"Rows   : {n:,}")
print(f"Size   : {size_mb:.1f} MB")

## Step 5 — Validation

Checks:
1. **Row counts** — compras: ~712k | pncp: ~619k
2. **cnpj_normalized integrity** — 14 chars, no punctuation, never null
3. **Nulls on critical fields**
4. **Date integrity** — no nulls on mandatory dates
5. **Value integrity** — negative flags, zero counts
6. **Sample rows**

In [ ]:
# =============================================================================
# Step 5 — Validation
# =============================================================================

con_val = get_connection()

# Filtros idênticos ao COPY — garante consistência Bronze -> Silver
COMPRAS_FILTER = "length(niFornecedor) = 14 AND contratoExcluido = 'false'"
PNCP_FILTER = """
    tipoPessoa = 'PJ'
    AND (codigoPaisFornecedor = 'BRA' OR codigoPaisFornecedor IS NULL)
    AND length(niFornecedor) = 14
    AND dataAssinatura BETWEEN '2021-01-01' AND '2030-12-31'
"""

EXPECTED_COMPRAS = scalar(con_val, f"SELECT count(*) FROM read_parquet('{BD_COMPRAS}', hive_partitioning=true) WHERE {COMPRAS_FILTER}")
EXPECTED_PNCP    = scalar(con_val, f"SELECT count(*) FROM read_parquet('{BD_PNCP}') WHERE {PNCP_FILTER}")

print(f"Expected silver_compras : {EXPECTED_COMPRAS:,}")
print(f"Expected silver_pncp    : {EXPECTED_PNCP:,}")
print()

# ── silver_compras ────────────────────────────────────────────────────────────
suite_c = CheckSuite("silver_compras")
n_c     = scalar(con_val, f"SELECT count(*) FROM read_parquet('{SD_COMPRAS}/**/*.parquet')")

suite_c.add("Row count matches Bronze PJ scope", n_c == EXPECTED_COMPRAS, f"{n_c:,} rows (expected {EXPECTED_COMPRAS:,})")

nulls     = scalar(con_val, f"SELECT count(*) FROM read_parquet('{SD_COMPRAS}/**/*.parquet') WHERE cnpj_normalized IS NULL")
wrong_len = scalar(con_val, f"SELECT count(*) FROM read_parquet('{SD_COMPRAS}/**/*.parquet') WHERE length(cnpj_normalized) != 14")
has_punct = scalar(con_val, f"""
    SELECT count(*) FROM read_parquet('{SD_COMPRAS}/**/*.parquet')
    WHERE cnpj_normalized LIKE '%.%'
       OR cnpj_normalized LIKE '%/%'
       OR cnpj_normalized LIKE '%-%'
""")
suite_c.add("cnpj_normalized — no nulls",       nulls     == 0, f"{nulls} nulls")
suite_c.add("cnpj_normalized — length = 14",    wrong_len == 0, f"{wrong_len} wrong length")
suite_c.add("cnpj_normalized — no punctuation", has_punct == 0, f"{has_punct} with punctuation")

for col in ["cnpj_basico", "numero_contrato", "data_vigencia_inicial", "valor_global"]:
    n_null = scalar(con_val, f"SELECT count(*) FROM read_parquet('{SD_COMPRAS}/**/*.parquet') WHERE {col} IS NULL")
    suite_c.add(f"{col} — no nulls", n_null == 0, f"{n_null} nulls")

negativos = scalar(con_val, f"SELECT count(*) FROM read_parquet('{SD_COMPRAS}/**/*.parquet') WHERE _valor_negativo = TRUE")
suite_c.add("_valor_negativo count > 0", negativos > 0, f"{negativos:,} contractual reductions (expected > 0)")

suite_c.report()
suite_c.assert_all_pass()

# Sample rows
print("\n  Sample rows — silver_compras:")
sample = con_val.execute(f"""
    SELECT cnpj_normalized, nome_fornecedor, nome_orgao,
           data_vigencia_inicial, data_vigencia_final, valor_global
    FROM read_parquet('{SD_COMPRAS}/**/*.parquet')
    LIMIT 4
""").fetchall()
for row in sample:
    def fmt(v): return (str(v) if v is not None else "NULL")[:20]
    print("    " + "  ".join(fmt(v) for v in row))

print()

# ── silver_pncp ───────────────────────────────────────────────────────────────
suite_p = CheckSuite("silver_pncp")
n_p     = scalar(con_val, f"SELECT count(*) FROM read_parquet('{SD_PNCP}')")

suite_p.add("Row count matches Bronze PJ scope", n_p == EXPECTED_PNCP, f"{n_p:,} rows (expected {EXPECTED_PNCP:,})")

nulls     = scalar(con_val, f"SELECT count(*) FROM read_parquet('{SD_PNCP}') WHERE cnpj_normalized IS NULL")
wrong_len = scalar(con_val, f"SELECT count(*) FROM read_parquet('{SD_PNCP}') WHERE length(cnpj_normalized) != 14")
has_punct = scalar(con_val, f"""
    SELECT count(*) FROM read_parquet('{SD_PNCP}')
    WHERE cnpj_normalized LIKE '%.%'
       OR cnpj_normalized LIKE '%/%'
       OR cnpj_normalized LIKE '%-%'
""")
suite_p.add("cnpj_normalized — no nulls",       nulls     == 0, f"{nulls} nulls")
suite_p.add("cnpj_normalized — length = 14",    wrong_len == 0, f"{wrong_len} wrong length")
suite_p.add("cnpj_normalized — no punctuation", has_punct == 0, f"{has_punct} with punctuation")

for col in ["cnpj_basico", "numero_controle_pncp", "data_assinatura", "valor_global"]:
    n_null = scalar(con_val, f"SELECT count(*) FROM read_parquet('{SD_PNCP}') WHERE {col} IS NULL")
    suite_p.add(f"{col} — no nulls", n_null == 0, f"{n_null} nulls")

invalidos = scalar(con_val, f"SELECT count(*) FROM read_parquet('{SD_PNCP}') WHERE _esfera_valida = FALSE")
suite_p.add("_esfera_valida=FALSE count >= 0", invalidos >= 0, f"{invalidos:,} records with esfera='N' (flagged, not removed)")

suite_p.report()
suite_p.assert_all_pass()

# Sample rows
print("\n  Sample rows — silver_pncp:")
sample = con_val.execute(f"""
    SELECT cnpj_normalized, nome_fornecedor, orgao_razao_social,
           data_assinatura, data_vigencia_fim, valor_global, orgao_esfera
    FROM read_parquet('{SD_PNCP}')
    LIMIT 4
""").fetchall()
for row in sample:
    def fmt(v): return (str(v) if v is not None else "NULL")[:20]
    print("    " + "  ".join(fmt(v) for v in row))

print()
con_val.close()

total_elapsed = (datetime.now(timezone.utc) - STARTED_AT).total_seconds()
print(f"Notebook completed in {total_elapsed:.0f}s ({total_elapsed/60:.1f} min)")